# TFL Generation Agent — Final Project Demo
**Standalone SAP → candidate TFL specification.** One notebook, no previous notebooks, experiment ZIPs or evaluation labels. Read the SAP from Google Drive, parse the specified body, build local BGE/BM25 retrieval, select a generic library template, extract evidence-grounded candidate statistics, validate and align. Display a report and copy unchanged library reference DOCX files. This is not an adapted, clinically approved final TFL shell. Table targets only.

**How to run:** put SAP_001_00.pdf in the existing Drive project folder, verify pages 4–21 are the SAP body, edit TARGET_TITLE to the actual table title if needed. For real inference set AUTHORIZE_LIVE=True after confirming permission to share retrieved passages with DeepSeek and approving at most two new completion attempts for this SAP/target/version. Add DEEPSEEK_API_KEY in Colab Secrets. Then Run all. Default is a local dry run, or an exact replay of this demo's own previously saved responses; no historical experiment artifacts are used.

In [ ]:
#@title 1. Demo configuration
TARGET_TITLE = "Patients Disposition - All Patients" #@param {type:"string"}
AUTHORIZE_LIVE = False #@param {type:"boolean"}
SAP_BODY_PAGE_RANGE = (4, 21)
SAP_FILENAME = "SAP_001_00.pdf"
MODEL = "deepseek-flash"
MAX_NEW_ATTEMPTS = 2
# The target title above is an editable example, not a title verified in this new PDF.
assert isinstance(TARGET_TITLE, str) and TARGET_TITLE.strip()
TARGET_TITLE = TARGET_TITLE.strip()
print("Target:", TARGET_TITLE)
print("Physical PDF body pages:", SAP_BODY_PAGE_RANGE)
print("New completion attempt cap:", MAX_NEW_ATTEMPTS if AUTHORIZE_LIVE else 0)


In [ ]:
#@title 2. Load frozen Agent code and SAP from Google Drive
from pathlib import Path
import sys, subprocess, importlib, shutil, json, time, csv, os
from datetime import datetime, timezone
from collections import Counter
from html import escape
from IPython.display import display, HTML
from google.colab import drive

REPO = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
BRANCH = "simplified-rag-v3"
PIN = "a22cc8dc65402d07e6f9865d4c98809e4f535d26"
ROOT = Path("/content/tfl-final-demo-v2")
if not ROOT.exists():
    subprocess.run(["git","clone","--branch",BRANCH,REPO,str(ROOT)],check=True)
if subprocess.run(["git","-C",str(ROOT),"cat-file","-e",PIN+"^{commit}"],capture_output=True).returncode:
    subprocess.run(["git","-C",str(ROOT),"fetch","origin",BRANCH],check=True)
subprocess.run(["git","-C",str(ROOT),"checkout","--detach",PIN],check=True)
assert subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()==PIN
subprocess.run([sys.executable,"-m","pip","install","-q","-e",f"{ROOT}[retrieval,test]"],check=True)
SRC=ROOT/"src"
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=="tfl_agent" or name.startswith("tfl_agent."):
        del sys.modules[name]
for p in (SRC/"tfl_agent").rglob("__pycache__"):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
from tfl_agent import TFLAgent
from tfl_agent.agent.unified_pipeline import (ResponseExecutor, finish_prepared,
    sha_file, digest, write_json, read_json, summarize)
from tfl_agent.tools.sap_hybrid import LocalEncoder
import pymupdf, torch

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive",timeout_ms=180000)
BASE=Path("/content/drive/MyDrive/Colab Notebooks/agent/midterm demo")
assert BASE.is_dir(), f"Missing project directory: {BASE}"
SAP=BASE/SAP_FILENAME
if not SAP.is_file():
    raise FileNotFoundError(f"Place the SAP PDF at exactly: {SAP}")
LIBRARY=ROOT/"generic_template_library"
assert LIBRARY.is_dir()
with pymupdf.open(SAP) as pdf:
    first,last=SAP_BODY_PAGE_RANGE
    assert 1<=first<=last<=len(pdf), f"Invalid physical range for {len(pdf)}-page PDF"
    assert any(pdf[p-1].get_text().strip() for p in range(first,last+1)), "No selectable text in SAP body"
    print("PDF pages:",len(pdf),"| selected body:",first,last)
    print("First body page excerpt:",pdf[first-1].get_text()[:350])
    print("Last body page excerpt:",pdf[last-1].get_text()[-350:])
print("SAP:",SAP)
print("Code revision:",PIN)
print("Local embedding device:","cuda" if torch.cuda.is_available() else "cpu")


## One automatic agent execution
The following cell runs the existing TFLAgent preparation and finish methods once, using two separate evidence-grounded requests. Matching receives primary passages and approved template structures; extraction receives primary plus supplementary passages, not template defaults. Exact fingerprints bind the demo's own persistent response ledger. A rerun can reuse those responses but does not read any past experiment ZIP. Do not delete or reset the ledger; do not run concurrent runtimes for the same SAP/target/version. If a request was reserved and interrupted, it is not automatically retried. A dry run does not send SAP passages to DeepSeek.

In [ ]:
#@title 3. Run the complete candidate pipeline
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DEST=BASE/"Final_Agent_Demo_Outputs"
DEST.mkdir(parents=True,exist_ok=True)
OUT=DEST/RUN_ID
OUT.mkdir(exist_ok=False)
SAP_HASH=sha_file(SAP)
case_identity={"commit":PIN,"sap_sha256":SAP_HASH,"target_title":TARGET_TITLE,
               "body_pages":list(SAP_BODY_PAGE_RANGE),"model":MODEL}
case_key=digest(case_identity)
CACHE=BASE/"Final_Agent_Demo_Cache"/case_key
write_json(OUT/"Input.json",{**case_identity,"sap_filename":SAP.name,
                           "authorized_new_attempts":MAX_NEW_ATTEMPTS if AUTHORIZE_LIVE else 0})
stage="initialize_agent"
plan=result=agent=executor=None
started=time.perf_counter()
try:
    agent=TFLAgent(library_path=LIBRARY,model=MODEL,
                   encoder=LocalEncoder(device="cuda" if torch.cuda.is_available() else "cpu"))
    print("Library:",agent.library.summary(),flush=True)
    stage="parse_chunk_retrieve"
    t=time.perf_counter()
    plan=agent.prepare(SAP,TARGET_TITLE,body_pages=SAP_BODY_PAGE_RANGE)
    preparation_seconds=time.perf_counter()-t
    assert plan["sap_sha256"]==SAP_HASH
    assert plan["source_chunks"], "No retrieved SAP evidence"
    assert all(first<=p<=last for c in plan["source_chunks"] for p in c["pages"])
    write_json(OUT/"Prepared_Plan.json",plan)
    identity={**case_identity,
              "matching_fingerprint":plan["matching_request"]["fingerprint"],
              "specification_fingerprint":plan["specification_request"]["fingerprint"]}
    stage="configure_executor"
    manifest=CACHE/"Request_Identity.json"
    if CACHE.exists():
        if not manifest.is_file() or read_json(manifest)!=identity:
            raise ValueError("Demo ledger identity differs; inspect cache, do not reset it.")
    elif AUTHORIZE_LIVE:
        CACHE.mkdir(parents=True,exist_ok=False)
        with manifest.open("x",encoding="utf-8") as f:
            json.dump(identity,f,ensure_ascii=False,indent=2)
            f.flush()
            os.fsync(f.fileno())
    if AUTHORIZE_LIVE:
        from google.colab import userdata
        try:
            key=userdata.get("DEEPSEEK_API_KEY").strip()
            if not key or any(c.isspace() for c in key):
                raise ValueError("Invalid API key format")
        except Exception:
            raise RuntimeError("Set a valid DEEPSEEK_API_KEY in Colab Secrets and enable notebook access.") from None
        executor=ResponseExecutor(cache_dir=CACHE,authorized=True,
                                  max_new_attempts=MAX_NEW_ATTEMPTS,api_key=key)
        del key
    else:
        executor=ResponseExecutor(cache_dir=CACHE if CACHE.exists() else None)
    agent.executor=executor
    print("Retrieved:",len(plan["retrieval"]["primary_ids"]),"primary +",
          len(plan["retrieval"]["supplementary_ids"]),"supplementary chunks",flush=True)
    stage="matching_extraction_validation_alignment"
    t=time.perf_counter()
    result=finish_prepared(plan,agent.executor)
    finish_seconds=time.perf_counter()-t
    assert not result["generation_authorized"]
    assert result["confirmed_template"] is None and result["confirmed_preset"] is None
    assert result["new_attempts"] <= (MAX_NEW_ATTEMPTS if AUTHORIZE_LIVE else 0)
    write_json(OUT/"Agent_Result.json",result)
    write_json(OUT/"Candidate_Specification.json",result["specification"])
    write_json(OUT/"Retrieval_Evidence.json",result["source_chunks"])
    summary=summarize([result])
    summary.update(code_commit=PIN,target=TARGET_TITLE,sap_sha256=SAP_HASH,
                   body_pages=list(SAP_BODY_PAGE_RANGE),
                   recommended_template=result["recommended_template"],
                   proposed_preset=result["proposed_preset"],
                   stage_status=result["stage_status"],
                   matching_protocol_valid=result["matching_protocol_valid"],
                   preparation_seconds=round(preparation_seconds,3),
                   finish_seconds=round(finish_seconds,3),
                   total_seconds=round(time.perf_counter()-started,3),
                   responses={k:{q:v.get(q) for q in ("status","cache_source","finish_reason","usage","reason","error_type","http_status")}
                              for k,v in result["responses"].items()},
                   demo_status="candidate_pipeline_complete" if result["complete_candidate_pipeline"]
                              else "partial_or_dry_run",
                   required_field_gaps=len(result["required_field_gaps"]))
    write_json(OUT/"Summary.json",summary)
    print("Demo status:",summary["demo_status"],"| new attempts:",result["new_attempts"])
    print("Results folder:",OUT)
except Exception as exc:
    write_json(OUT/"Failure.json",{"stage":stage,"error_type":type(exc).__name__,
                                   "new_attempts":executor.new_attempts if executor else 0})
    print("Failed stage:",stage,"|",type(exc).__name__,"| artifacts:",OUT)
    raise


## Final presentation
The report separates raw template recommendation, aligned candidate, retained claims, review items, and unknown information. Missing values are never filled from library defaults. The reference DOCX is copied only after the candidate pipeline is complete; if no preset is proposed, all reference options for the recommended family are listed without selecting a study-specific preset. These are unchanged generic examples, not generated target-study shells.

In [ ]:
#@title 4. Display and save the presentation-ready results
import pandas as pd
from docx import Document

def table_html(headers, rows):
    def cell(x):
        if x is None: x="—"
        if isinstance(x,(dict,list,tuple)):
            x=json.dumps(x,ensure_ascii=False)
        return escape(str(x))
    return ("<table><thead><tr>"+"".join("<th>"+cell(h)+"</th>" for h in headers)+
            "</tr></thead><tbody>"+"".join("<tr>"+"".join("<td>"+cell(x)+"</td>" for x in row)+"</tr>" for row in rows)+"</tbody></table>")

assert result is not None, "Run the Agent cell first."
lookup={c["chunk_id"]:c for c in result["source_chunks"]}
claim_rows=[]
for c in result["claim_audit"]:
    pages=sorted({s["page"] for s in c.get("source_spans",[])})
    claim_rows.append({"field":c.get("field"),"value":c.get("value"),
                       "route":c.get("rule_route"),"pages":pages or ("title" if c.get("source")=="title" else []),
                       "exact_quote":c.get("exact_quote"),"chunk_id":c.get("chunk_id"),
                       "exact_match":c.get("exact_match"),"risk_flags":c.get("risk_flags",[])})
fields=[]
for f,v in result["specification"]["fields"].items():
    fields.append({"field":f,"status":v["status"],"candidate_values":" | ".join(v["values"]),
                   "retained_claim_indices":v["claim_indices"],"other_claim_indices":v["other_claim_indices"]})
pd.DataFrame(fields).to_csv(OUT/"Specification_Fields.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(claim_rows,columns=["field","value","route","pages","exact_quote","chunk_id","exact_match","risk_flags"]).to_csv(OUT/"Claim_Audit.csv",index=False,encoding="utf-8-sig")

matching=result["matching"] or {}
rec=result["recommended_template"]
reference_records=[]
reference_notes=[]
reference_preview=None
if result["complete_candidate_pipeline"] and rec in agent.library.templates:
    index_entry=agent.library.templates[rec]
    write_json(OUT/"Recommended_Template_Definition.json",agent.library.template(rec))
    refs=index_entry.get("reference_shells",[])
    proposed=result["proposed_preset"]
    # A proposed preset is still unvalidated. Without one, list all family references.
    selected=[r for r in refs if r["preset_id"]==proposed] if proposed else refs
    if not selected:
        reference_notes.append("No matching reference DOCX listed in the approved index.")
    for ref in selected:
        try:
            path=agent.library.safe_path(ref["docx"])
            if sha_file(path)!=ref["sha256"]:
                raise ValueError("Reference DOCX checksum mismatch")
            dest=OUT/"Library_Reference_Shells"/path.name
            dest.parent.mkdir(exist_ok=True)
            shutil.copy2(path,dest)
            assert sha_file(dest)==ref["sha256"]
            reference_records.append({"preset_id":ref["preset_id"],"file":str(dest.relative_to(OUT)),
                                      "sha256":ref["sha256"],"status":"unchanged_generic_reference"})
            if reference_preview is None:
                document=Document(dest)
                if document.tables:
                    table=document.tables[0]
                    sample=[[cell.text for cell in row.cells] for row in list(table.rows)[:12]]
                    if sample:
                        reference_preview=(dest.name,sample)
        except Exception as exc:
            reference_notes.append(f"{ref.get('preset_id')}: {type(exc).__name__}")
write_json(OUT/"Reference_Shells.json",{"references":reference_records,"notes":reference_notes,
                                      "study_specific_shell_generated":False})
stats=[
    ("Input PDF",SAP.name),("Target TFL",TARGET_TITLE),("Physical body pages",str(SAP_BODY_PAGE_RANGE)),
    ("SAP blocks",result["retrieval"]["block_count"]),("SAP chunks",result["retrieval"]["chunk_count"]),
    ("Primary retrieved",len(result["retrieval"]["primary_ids"])),
    ("Supplementary retrieved",len(result["retrieval"]["supplementary_ids"])),
    ("Original matching",matching.get("recommended_template")),
    ("Final candidate",rec),("Matching protocol valid",result["matching_protocol_valid"]),
    ("Alignment decision",result["stage_status"]["alignment"]),
    ("Proposed preset (unvalidated)",result["proposed_preset"]),
    ("Candidate fields",sum(f["status"]=="candidate" for f in fields)),
    ("New model attempts",result["new_attempts"]),("Preparation / inference seconds",f"{preparation_seconds:.2f} / {finish_seconds:.2f}")
]
sections=[
    "<h1>TFL Generation Agent</h1><p class='lead'>SAP → Hybrid RAG → Generic Template Matching → Candidate Specification → Alignment</p>",
    "<div class='notice'>"+escape("Candidate pipeline completed; clinical validation and target-shell rendering remain outstanding." if result["complete_candidate_pipeline"] else "Partial / dry run: no complete candidate pipeline. See response statuses below.")+"</div>",
    "<h2>1. Pipeline summary</h2>"+table_html(["Metric","Result"],stats),
    "<h2>2. Library matching</h2>"+table_html(["Rank","Template"],list(enumerate(matching.get("rank_prefix",[]),1)))+
      "<p>"+escape(str(matching.get("reason") or "No complete matching response."))+"</p>"+
      "<p>Protocol issues: "+escape(str(matching.get("issues",[])))+"</p>",
    "<h2>3. Candidate specification</h2>"+table_html(["Field","Status","Candidate values"],[(f["field"],f["status"],f["candidate_values"] or "—") for f in fields]),
    "<h2>4. Evidence and validation</h2>"+table_html(["Field","Value","Route","PDF page(s)","Exact SAP / title quote"],[(r["field"],r["value"],r["route"],r["pages"],r["exact_quote"]) for r in claim_rows]),
    "<h2>5. Unbound library requirements</h2>"+table_html(["Required path","Status"],[(g["path"],g["status"]) for g in result["required_field_gaps"]]),
    "<h2>6. Retrieved passages</h2>"
]
primary=set(result["retrieval"]["primary_ids"])
for c in result["source_chunks"]:
    label=f"{'Primary' if c['chunk_id'] in primary else 'Supplementary'} | pages {c['pages']} | {c['chunk_id']}"
    sections.append("<details><summary>"+escape(label)+"</summary><pre>"+escape(c["text"])+"</pre></details>")
sections.append("<h2>7. Reference shells — generic examples only</h2>"+
                table_html(["Preset","Unchanged reference file"],[(r["preset_id"],r["file"]) for r in reference_records])+
                "<p>"+escape(" ".join(reference_notes))+"</p>")
if reference_preview:
    name,rows=reference_preview
    sections.append("<p>Illustrative preview (first table, up to 12 rows): "+escape(name)+"</p>"+
                    table_html(["Column "+str(i+1) for i in range(len(rows[0]))],rows))
sections.append("<h2>8. Model response audit</h2>"+table_html(["Module","Response status","Source","Token usage"],[(k,v.get("status"),v.get("cache_source"),v.get("usage")) for k,v in result["responses"].items()]))
sections.append("<p><strong>Limitations:</strong> Candidate only; no confirmed template or preset, no clinical approval, no fabricated statistical values, no generated study-specific DOCX. Unknown fields are not proof of non-applicability. Reference shell preview is not the generated output.</p>")
css="""<style>
.demo-report{font-family:Arial,sans-serif;color:#243449;background:#fff;padding:24px;line-height:1.45}
.demo-report h1{font-size:27px;color:#123456;margin:0 0 8px}
.demo-report h2{font-size:18px;margin-top:25px;border-bottom:1px solid #d3dee7;padding-bottom:6px}
.demo-report .lead{color:#475d73}.demo-report .notice{padding:12px;border-left:4px solid #bc8125;background:#fff6e7}
.demo-report table{border-collapse:collapse;width:100%;font-size:13px;margin:8px 0 12px}
.demo-report th,.demo-report td{padding:8px;border:1px solid #d5dee7;text-align:left;vertical-align:top;overflow-wrap:anywhere}
.demo-report th{background:#edf2f6}.demo-report details{margin:5px 0}.demo-report pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f6f8fa;padding:8px}
</style>"""
body=css+"<div class='demo-report'>"+"".join(sections)+"</div>"
(OUT/"Demo_Report.html").write_text("<!doctype html><html lang='en'><head><meta charset='utf-8'><title>TFL Agent Demo</title></head><body>"+body+"</body></html>",encoding="utf-8")
display(HTML(body))
# Zip is a new optional output, never an input dependency.
archive=Path(shutil.make_archive(str(DEST/RUN_ID)+"_demo","zip",root_dir=OUT))
print("Report:",OUT/"Demo_Report.html")
print("Output archive:",archive)
print("Keep demo outputs private unless authorized to share SAP excerpts.")


**Final presentation checklist:** Show the input filename and target title, primary/supplementary evidence, raw and aligned template, candidate fields with quotations, unresolved requirements and the reference example. If no complete model result, show the partial/dry status honestly. The demo is separate from formal external evaluation, uses no ground truth, and does not claim accuracy on this new SAP.